# Olist E-Commerce Data
**Domain:** Financial Optimization, Logistics Compliance, and People Analytics Architecture

**Where the data goes:** SQL Server (Star Schema) 

This notebook automates Extraction, Transformation, and Loading (ETL) pipeline, converting raw transactional datasets into an analytics-ready relational data warehouse.

In [1]:
import sys
import os
import glob
import pandas as pd
import urllib.parse
from sqlalchemy import create_engine

# Telling Python to look into the 'scripts' folder to find our config file
sys.path.append(os.path.abspath("../scripts"))
import config

# Setting up the folder path where our raw Olist CSVs are stored
folder_path = "../data_raw"

# Finding all files ending with .csv in that folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Dictionary to hold all our pandas DataFrames
datasets = {}

print("--- Starting data ingestion ---")
for file in csv_files:
    # Extracting the file name without the '.csv' extension to use as a key
    table_name = os.path.basename(file).replace(".csv", "")
    
    # Reading the CSV file and saving it into our dictionary
    datasets[table_name] = pd.read_csv(file)
    print(f"Dataset '{table_name}' loaded successfully.")

--- Starting data ingestion ---
Dataset 'olist_customers_dataset' loaded successfully.
Dataset 'olist_geolocation_dataset' loaded successfully.
Dataset 'olist_orders_dataset' loaded successfully.
Dataset 'olist_order_items_dataset' loaded successfully.
Dataset 'olist_order_payments_dataset' loaded successfully.
Dataset 'olist_order_reviews_dataset' loaded successfully.
Dataset 'olist_products_dataset' loaded successfully.
Dataset 'olist_sellers_dataset' loaded successfully.


## Step 1: Taking a First Look at the Data
Before changing anything, we need to know exactly what we are working with. In this section, I am running a quick scan across all the tables to check data types, row counts, and where the missing values (nulls) are hiding. This helps us plan our cleaning phase without any guesswork.

In [2]:
# Iterates through loaded tables to display structural info
for table_name, df in datasets.items():
    print(f"\n==================== {table_name} ====================")
    df.info()


==================== olist_customers_dataset ====================
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB

==================== olist_geolocation_dataset ====================
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geo

In [3]:
# Iterates through loaded tables to display structural null distributions
for table_name, df in datasets.items():
    print(f"\n==================== {table_name} ====================")
    print(pd.DataFrame({"Data_Type": df.dtypes, "Null_Count": df.isnull().sum()}))


==================== olist_customers_dataset ====================
                         Data_Type  Null_Count
customer_id                    str           0
customer_unique_id             str           0
customer_zip_code_prefix     int64           0
customer_city                  str           0
customer_state                 str           0

==================== olist_geolocation_dataset ====================
                            Data_Type  Null_Count
geolocation_zip_code_prefix     int64           0
geolocation_lat               float64           0
geolocation_lng               float64           0
geolocation_city                  str           0
geolocation_state                 str           0

==================== olist_orders_dataset ====================
                              Data_Type  Null_Count
order_id                            str           0
customer_id                         str           0
order_status                        str           0
order_purc

In [4]:
# Iterates through loaded tables to display structural describe
for table_name, df in datasets.items():
    print(f"\n==================== {table_name} ====================")
    print(df.describe())


==================== olist_customers_dataset ====================


       customer_zip_code_prefix
count              99441.000000
mean               35137.474583
std                29797.938996
min                 1003.000000
25%                11347.000000
50%                24416.000000
75%                58900.000000
max                99990.000000

==================== olist_geolocation_dataset ====================
       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng
count                 1.000163e+06     1.000163e+06     1.000163e+06
mean                  3.657417e+04    -2.117615e+01    -4.639054e+01
std                   3.054934e+04     5.715866e+00     4.269748e+00
min                   1.001000e+03    -3.660537e+01    -1.014668e+02
25%                   1.107500e+04    -2.360355e+01    -4.857317e+01
50%                   2.653000e+04    -2.291938e+01    -4.663788e+01
75%                   6.350400e+04    -1.997962e+01    -4.376771e+01
max                   9.999000e+04     4.506593e+01     1.211054e+02

==================== 

## Step 2: Cleaning Things Up
Real-world data is almost always messy. To make sure our final dashboards don't show wrong numbers, we need to fix a few things first. Here, I am running a quick loop to drop any exact duplicate rows so we don't double-count revenue by mistake. I am also filling in the blank product categories with an 'uninformed' label so our charts look clean later on.

In [5]:
# 1. Dropping exact duplicates across all tables to avoid double-counting revenue
for table_name in datasets:
    datasets[table_name] = datasets[table_name].drop_duplicates()
print("Data integrity check complete: Exact duplicates removed.")

# 2. Filling missing values in product categories so they don't look blank in Power BI
datasets["olist_products_dataset"]["product_category_name"] = (
    datasets["olist_products_dataset"]["product_category_name"].fillna("uninformed")
)
print("Product category clean-up complete.")

Data integrity check complete: Exact duplicates removed.
Product category clean-up complete.


## Step 3: Creating Our Business Metrics
This is where we turn raw data into useful insights. To help both the Logistics and People Analytics teams, I built a custom Python function to calculate three key things:
1. **delay_days:** Exactly how many days a delivery was late compared to the promise made to the customer.
2. **is_delayed:** A simple 1 or 0 flag to quickly filter late orders.
3. **seller_dispatch_days:** How long a seller took to hand over the product to the delivery company. For our People Analytics page, this helps us measure operational efficiency and potential stress points in our partner network.

In [6]:
def clean_and_engineer_orders(df_orders):
    """Parses date columns and calculates logistics and operational KPIs."""
    df = df_orders.copy()

    # Changing date columns from text format to datetime objects so we can do math with them
    date_columns = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]

    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # Cleaning up whitespaces and lowercasing order status values
    if "order_status" in df.columns:
        df["order_status"] = df["order_status"].str.strip().str.lower()

    # Metric 1: Counting how many days a delivery was late (actual vs estimated)
    df["delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
    df["delay_days"] = df["delay_days"].fillna(0).astype(int)

    # Metric 2: Creating a 1 or 0 flag to quickly filter late orders
    df["is_delayed"] = (df["delay_days"] > 0).astype(int)

    # Metric 3: Tracking how many days the seller took to ship the product (for People Analytics)
    df["seller_dispatch_days"] = (df["order_delivered_carrier_date"] - df["order_approved_at"]).dt.days
    df["seller_dispatch_days"] = df["seller_dispatch_days"].fillna(0).astype(int)

    return df

# Applying the changes back to our orders dataset
datasets["olist_orders_dataset"] = clean_and_engineer_orders(datasets["olist_orders_dataset"])
print("New metrics successfully created.")

New metrics successfully created.


## Step 4: Loading Everything into SQL Server
Now that the data is clean and our new metrics are ready, it's time to save everything into the database. I am connecting to SQL Server using local Windows Authentication. Since some tables (like geolocation) are huge, the code is set up to upload the data in batches of 5,000 rows. This keeps everything smooth and prevents the system from running out of memory or crashing.

In [7]:
# Building the SQL Server connection string using local Windows Authentication
string_conexao = (
    f"mssql+pyodbc://@{config.DB_SERVER}/{config.DB_NAME}?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# Enabling fast_executemany to speed up the database upload significantly
engine = create_engine(string_conexao, fast_executemany=True)

# Uploading our clean tables to SQL Server in smaller batches to avoid crashing the memory
for table_name, df in datasets.items():
    print(f"Uploading '{table_name}' ({len(df):,} rows)...")
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=5000,  # Sending 5,000 rows at a time
    )
    print(f"Table '{table_name}' saved to SQL Server.\n")

print("All datasets uploaded! We are ready for Power BI!")

Uploading 'olist_customers_dataset' (99,441 rows)...
Table 'olist_customers_dataset' saved to SQL Server.

Uploading 'olist_geolocation_dataset' (738,332 rows)...
Table 'olist_geolocation_dataset' saved to SQL Server.

Uploading 'olist_orders_dataset' (99,441 rows)...
Table 'olist_orders_dataset' saved to SQL Server.

Uploading 'olist_order_items_dataset' (112,650 rows)...
Table 'olist_order_items_dataset' saved to SQL Server.

Uploading 'olist_order_payments_dataset' (103,886 rows)...
Table 'olist_order_payments_dataset' saved to SQL Server.

Uploading 'olist_order_reviews_dataset' (99,224 rows)...
Table 'olist_order_reviews_dataset' saved to SQL Server.

Uploading 'olist_products_dataset' (32,951 rows)...
Table 'olist_products_dataset' saved to SQL Server.

Uploading 'olist_sellers_dataset' (3,095 rows)...
Table 'olist_sellers_dataset' saved to SQL Server.

All datasets uploaded! We are ready for Power BI!
